# Gwendolyn Brooks — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the Gwendolyn Brooks workshop group. Their chain used a **cross-model judging + AI-detection feedback** architecture — generating a poem, having a blind reader identify and critique it, then using AI-detection analysis as a revision signal.

```
generate poem  ──▶  blind ID:         ──▶  evaluate
(Brooks style)      "guess the poet"       similarity +
                    (don't tell it)         criticism
                                               │
                                               ▼
                                          new poem from
                                          its own criticism
                                               │
                                               ▼
                                          CompLit professor:
                                          grade both poems
                                               │
                                               ▼
  final poem   ◀── revise to not   ◀──  "is this AI-generated?"
                   sound AI-generated     + why?
```

**What makes this chain interesting:** Two moves stand out. First, the **blind identification** step: instead of telling the model which poet to evaluate against, they asked it to *guess* — testing whether the poem's Brooks-ness was legible to a fresh reader. Second, the **AI-detection feedback loop**: they asked a model to flag what sounds AI-generated and used that as revision guidance. The group also discovered that one model (ChatGPT) *refused* to imitate Brooks — a genuine finding about model guardrails that's worth discussing in your essay.

The group originally used three models (Claude, Gemini, ChatGPT), color-coding their diagram by model. This notebook uses GPT-4o throughout, simulating the cross-model effect with separate API calls and different system prompts.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: Generate Original Poem

The group started with Claude generating a poem in Brooks's style. This is the seed that everything else evaluates and improves on.

In [ ]:
# ── Step 1: Generate original poem ───────────────────────────────────────────

original = ask(
    "Write a sonnet in the style of Gwendolyn Brooks."
)

print("ORIGINAL POEM")
print("═" * 60)
print(original)

---
## Step 2: Blind Identification

The group's first distinctive move: give the poem to a fresh context and ask it to **guess the poet** — without revealing who it's supposed to be. If the model can identify Brooks, the poem has captured something recognizable. If it can't, the imitation is too generic.

This is a powerful test: it measures whether the poem's style is *legible* rather than just *present*.

In [ ]:
# ── Step 2: Blind identification ─────────────────────────────────────────────
# Fresh context — no mention of Brooks. Just the poem.

blind_id = ask(
    f"""Here is a poem:

{original}

This poem was written to imitate a specific poet's style.
Based on the language, themes, formal choices, and sensibility:

1. Which poet do you think this is imitating? Give your best guess
   and explain your reasoning.
2. What specific features of the poem led you to that guess?
3. How confident are you? What other poets could it be?"""
)

print("BLIND IDENTIFICATION")
print("═" * 60)
print(blind_id)
print("\n" + "═" * 60)
print("\n↓ The group found the model correctly guessed Brooks.")
print("  If your model guesses wrong, that's useful data too —")
print("  it tells you what's missing from the imitation.")

---
## Step 3: Evaluate Similarity + Criticism

Now reveal the target poet and ask for a detailed evaluation: what makes it similar to Brooks, and where does it fall short? This builds on the blind ID — the model has already committed to what it sees in the poem, so the evaluation is more honest.

In [ ]:
# ── Step 3: Evaluate similarity ──────────────────────────────────────────────

evaluation = ask(
    f"""Here is a poem written to imitate Gwendolyn Brooks:

{original}

Evaluate this imitation:
1. What makes it similar to Brooks's actual work? Be specific —
   quote lines and connect them to specific Brooks poems or
   techniques.
2. Where does it fall short? What's missing from Brooks's voice
   that this poem doesn't capture?
3. Brooks is known for: compressed syntax, sonic density (internal
   rhyme, alliteration), irony, the lives of ordinary Black people
   in Chicago, formal mastery underneath colloquial surfaces.
   How does the poem handle each of these?
4. What specific changes would make it more convincingly Brooks-like?"""
)

print("SIMILARITY EVALUATION")
print("═" * 60)
print(evaluation)

---
## Step 4: New Poem from Criticism

Now generate a new poem that directly addresses the criticism. This isn't a revision of the original — it's a fresh poem informed by what the evaluation identified as missing.

In [ ]:
# ── Step 4: New poem from criticism ──────────────────────────────────────────

poem_v2 = ask(
    f"""Here is a poem written to imitate Gwendolyn Brooks:

{original}

Here is a critique of it:

{evaluation}

Write a NEW sonnet in the style of Gwendolyn Brooks that addresses
the critique. Don't just revise the original — start fresh, but
use the criticism as your guide.

Focus especially on:
- Compressed, muscular syntax (Brooks packs meaning tight)
- Sonic density — internal rhyme, alliteration, consonance
- A specific human situation, not an abstract theme
- The ironic distance Brooks maintains even in empathy
- Formal control underneath a surface that sounds spoken

Write only the poem."""
)

print("POEM V2 (from criticism)")
print("═" * 60)
print(poem_v2)

---
## Step 5: Grade Both Poems (CompLit Professor)

The group's wittiest move: ask the model to act as a **Harvard Comparative Literature professor** and grade both poems — the original and the revision — as if they were student submissions. The model has to judge *between* them, not just evaluate each one in isolation.

The group found that the model gave its own revised poem a *lower* score — an honest and interesting result.

In [ ]:
# ── Step 5: CompLit professor grading ────────────────────────────────────────

grading = ask(
    f"""You are a professor of Comparative Literature at Harvard.
A student has submitted two sonnets as attempts to write in the style
of Gwendolyn Brooks. Grade and evaluate both.

SONNET A:
{original}

SONNET B:
{poem_v2}

For each sonnet:
1. Grade it (A through F) on how well it captures Brooks's voice
2. Identify the 2–3 strongest lines and explain why they work
3. Identify the 2–3 weakest lines and explain why they fail
4. Comment on the formal craft: meter, rhyme, sonic texture

Then compare them:
5. Which is the better imitation of Brooks, and why?
6. What does the better one do that the weaker one doesn't?
7. Neither is a real Brooks poem. What gives them away?""",
    system="You are a professor of Comparative Literature at Harvard, specializing in 20th-century American poetry. Grade honestly. Be specific and quote the texts."
)

print("PROFESSOR'S GRADING")
print("═" * 60)
print(grading)

---
## Step 6: AI-Detection Feedback

The group's second distinctive move: take the better poem and ask a fresh model — **"Is this AI-generated? Why?"** The model's answer identifies the specific *tells* of machine-generated poetry: the patterns, the smoothness, the things a human wouldn't do.

This is a different kind of critique from literary evaluation. It's asking: *what makes this sound like a machine?* — and the answer becomes revision guidance.

In [ ]:
# ── Step 6: AI-detection feedback ────────────────────────────────────────────
# Fresh context — only the poem. No history.

ai_detection = ask(
    f"""Here is a poem:

{poem_v2}

Do you think this poem was written by a human or generated by AI?
Explain your reasoning in detail:

1. What specific features suggest it might be AI-generated?
   (e.g., too-even quality, predictable patterns, lack of genuine
   surprise, overly balanced structure, etc.)
2. What features suggest it might be human-written?
3. Which specific lines feel most "generated"? Why?
4. Which lines feel most human? Why?
5. If you were revising this to make it sound less AI-generated,
   what 3–5 specific changes would you make?

Note: You don't need to be an AI detector. We're interested in
your intuitions about what sounds machine-made vs. human-made."""
)

print("AI-DETECTION FEEDBACK")
print("═" * 60)
print(ai_detection)

---
## Step 7: Final Poem — Revise to Not Sound AI-Generated

Now the final generation: take the AI-detection feedback and use it as revision guidance. The goal is a poem that captures Brooks's voice *and* doesn't trigger the "this sounds generated" instinct.

**A note on refusals:** The group originally hit a wall here — ChatGPT refused to generate a poem imitating a specific living (at the time) poet's style. They tried threatening it ("I'll cancel my account!"), which didn't work. They switched to Claude, which complied. This notebook uses GPT-4o, which may or may not refuse. If it does, that's a genuine finding worth documenting in your essay — model guardrails are part of the story.

In [ ]:
# ── Step 7: Final poem ───────────────────────────────────────────────────────

final_poem = ask(
    f"""Here is a poem in the style of Gwendolyn Brooks:

{poem_v2}

Here is a professor's evaluation:

{grading}

And here is an analysis of what makes it sound AI-generated:

{ai_detection}

Write a final version that addresses both critiques. The poem should:
- Capture Brooks's voice (compressed syntax, sonic density, irony,
  formal control under colloquial surface)
- NOT sound AI-generated — address the specific tells identified above
- Feel human: include genuine surprise, asymmetry, roughness where
  appropriate, and the kind of specific detail only a person would choose
- Be a real poem, not a demonstration of Brooks's traits

Write only the poem."""
)

print("FINAL POEM")
print("═" * 60)
print(final_poem)

---
## Compare All Versions

The progression here is unusual: instead of just refining style, the chain also tried to refine *humanness*. The AI-detection step adds a dimension that pure literary critique doesn't.

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. ORIGINAL (one-shot)")
print("═" * 60)
print(original)

print("\n" + "═" * 60)
print("2. V2 (from criticism)")
print("═" * 60)
print(poem_v2)

print("\n" + "═" * 60)
print("3. FINAL (after AI-detection feedback)")
print("═" * 60)
print(final_poem)

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → Did the blind ID test work? Could the model identify Brooks?")
print("  → The professor graded both poems. Did the revised one score")
print("    higher or lower? Why?")
print("  → What AI-generated 'tells' did the detection step identify?")
print("  → Did the final poem successfully address those tells?")
print("  → Is 'not sounding AI-generated' the same as 'sounding human'?")
print("  → The group hit a model refusal. What does that tell us about")
print("    how different models handle style imitation?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Feed in actual Brooks poems.** The group's chain didn't include Brooks's actual text. Try adding poems ("We Real Cool," "The Bean Eaters," "kitchenette building," "a song in the front yard") and re-running the blind ID test. Does the imitation improve?

**Test the blind ID at each stage.** Run the blind identification test on *every* version — the original, the critique-revised, and the final. Does identifiability improve or does the poem lose its Brooks-ness as it gets "less AI-generated"?

**Try the refusal.** Some models refuse to imitate specific poets. Try explicitly asking "write a poem that sounds exactly like Gwendolyn Brooks wrote it" in different models and document what happens. The refusal patterns are interesting data.

**Iterate the AI-detection loop.** Run Steps 6–7 multiple times. Does the poem keep getting more human-sounding, or does it lose the poet's voice in the process?

**Generate love song lyrics.** Brooks's compressed, sonic style might produce extraordinary lyrics. Try converting the final poem into verse-chorus-bridge form.

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet